# Model selection and My approach


###  What Manager(Priya) said

> *"Give us a list of customers who are about to leave, so we can call them first.*
> *Don't just send me an accuracy number. Tell me which model you chose, **why**, and how sure you are."*

### How I think about this task

### 🧠 How I think about this task

| A beginner would | I will instead |
|---|---|
| Train one model, report accuracy | Compare several models fairly and pick one with proof |
| Test on all the data | Lock a test set first and open it only once |
| Use the default 0.5 cut-off | Choose the cut-off using what mistakes cost the business |
| Say "the model is 80% accurate" | Say "it catches X of 100 churners and wastes Y calls" |

My job is **not** "train a model." My job is: **pick a model the business can trust, and prove it.**

## 🧭 My plan: 8 steps

Before I write any code, this is the order I will follow. For each step I ask myself one simple question.

| # | What I do | What I ask myself |
|---|---|---|
| 1 | **Decide how I will measure success** | What does the Retention team actually need? Catching churners, or just a high score? |
| 2 | **Split the data and lock the test set** | Have I kept some data aside that I will not touch until the end? |
| 3 | **Make a baseline** | If I build the dumbest possible model, what score does it get? My real model must beat this. |
| 4 | **Build a pipeline** | Am I cleaning the data in a way that doesn't accidentally peek at the test data? |
| 5 | **Compare a few models fairly** | Did every model get the same data, the same splits and the same scoring? |
| 6 | **Pick the best 2 and tune them** | Why waste time tuning models that already lost? |
| 7 | **Choose the cut-off** | At what probability do I say "this customer will leave"? Wrong guesses cost money, so I decide using cost. |
| 8 | **Test once and explain** | Does it still work on data it has never seen? Can I explain it to my manager in simple words? |

**My one golden rule:** I open the test data only once, at the very end. Every decision before that is made using the training data only.

## ⚙️ Setup
Imports and settings. I set `RANDOM_STATE` so Priya (or anyone) can run my code and get the same numbers.

In [37]:
from __future__ import annotations

import json
import logging
import time
import platform
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import joblib 
import numpy as np
import pandas as pd
import sklearn

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score,confusion_matrix,f1_score,
make_scorer,precision_score, recall_score,roc_auc_score)



from sklearn.model_selection import GridSearchCV, StratifiedKFold,cross_val_predict

from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier


log = logging.getLogger("churn.train")

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

from pathlib import Path
print(Path.cwd())

D:\Projects\notebooks\Churn_Prediction_Project\notebooks


# Look at the data first

first i look at the data, i never model blind .I check three thigs size, types , and how balanced the target is

In [38]:
@dataclass(frozen=True)
class Config:
    data_dir: Path = Path("data/processed")
    model_dir: Path = Path("models")
    report_dir: Path = Path("reports")
    random_state: int = 42
    cv_folds: int = 5
    primary_metric: str = "roc_auc"



PROJECT_DIR = Path(r"D:\Projects\notebooks\Churn_Prediction_Project")

cfg = Config(
    data_dir=PROJECT_DIR / "data" / "processed",
    model_dir=PROJECT_DIR / "models",
    report_dir=PROJECT_DIR / "reports",
)
print(cfg)


Config(data_dir=WindowsPath('D:/Projects/notebooks/Churn_Prediction_Project/data/processed'), model_dir=WindowsPath('D:/Projects/notebooks/Churn_Prediction_Project/models'), report_dir=WindowsPath('D:/Projects/notebooks/Churn_Prediction_Project/reports'), random_state=42, cv_folds=5, primary_metric='roc_auc')


In [39]:
from sklearn.model_selection import train_test_split

df = pd.read_csv(cfg.data_dir / "cleaned_churn.csv")
if df["Churn"].dtype == object:                  # "Yes"/"No" -> 1/0
    df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

X = df.drop(columns=["Churn"])
y = df["Churn"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Verify: ha split preprocessing notebook madhlya split sarkha ahe ka
saved_y = cfg.data_dir / "y_train.csv"
if saved_y.exists(): 
    old = pd.read_csv(saved_y).squeeze("columns")
    assert len(old) == len(y_tr) and (old.values == y_tr.values).all(), \
        "split differs from preprocessing notebook"

X_tr.to_csv(cfg.data_dir / "X_train.csv", index=False)
X_te.to_csv(cfg.data_dir / "X_test.csv", index=False)
y_tr.to_csv(cfg.data_dir / "y_train.csv", index=False)
y_te.to_csv(cfg.data_dir / "y_test.csv", index=False)
print("raw splits saved:", X_tr.shape, X_te.shape)

raw splits saved: (5625, 19) (1407, 19)


## Step 2: Load and validate the inputs

**Why:** `load_data` asserts matching lengths, identical columns and a 0/1 target, so a wrong file fails loudly here instead of quietly corrupting results. `load_preprocessor_recipe` calls `clone()`, which keeps the recipe but throws away what it learned on the full train set, so every CV fold learns its own scaling and encoding.

In [40]:
def load_data(cfg: Config):
    """Load the RAW splits produced by the preprocessing step and validate them."""
    X_train = pd.read_csv(cfg.data_dir / "X_train.csv")
    X_test = pd.read_csv(cfg.data_dir / "X_test.csv")
    y_train = pd.read_csv(cfg.data_dir / "y_train.csv").squeeze("columns")
    y_test = pd.read_csv(cfg.data_dir / "y_test.csv").squeeze("columns")

    # Fail fast: a silent data problem here would poison every later result.
    assert len(X_train) == len(y_train) and len(X_test) == len(y_test), "X/y length mismatch"
    assert list(X_train.columns) == list(X_test.columns), "train/test columns differ"
    assert set(y_train.unique()) <= {0, 1}, "target must be 0/1"
    assert X_train.select_dtypes(exclude="number").shape[1] > 0, "expected RAW (unencoded) features"

    log.info("train=%s churn=%.3f | test=%s churn=%.3f (test locked)",
             X_train.shape, y_train.mean(), X_test.shape, y_test.mean())
    return X_train, X_test, y_train, y_test


def load_preprocessor_recipe(cfg: Config):
    """Take the preprocessing DEFINITION, not its fitted state.

    preprocessor.pkl was fitted on the whole training set. If we reused that fitted
    object inside cross-validation, every validation fold would already be "known" to
    the scaler. clone() returns the same recipe with all learned state removed, so each
    CV fold re-learns its scaling/encoding from its own training part only.
    """
    return clone(joblib.load(cfg.model_dir / "preprocessor.pkl"))

In [41]:
X_train, X_test, y_train, y_test = load_data(cfg)   # test data is loaded but not used until Step 9
recipe = load_preprocessor_recipe(cfg)
recipe

2026-10-07 13:08:37,736 INFO train=(5625, 19) churn=0.266 | test=(1407, 19) churn=0.266 (test locked)


,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


In [42]:
def build_candidates(recipe, seed: int) -> dict:

    def pipe(estimator):
        return Pipeline([("prep", clone(recipe)), ("model", estimator)])

    return {
        "baseline": (
            pipe(DummyClassifier(strategy="prior")),
            {}
        ),
        "logreg": (
            pipe(LogisticRegression(max_iter=2000, class_weight="balanced")),
            {"model__C": [0.01, 0.1, 1, 10]}   # 4 combinations
        ),
        "decision_tree": (
            pipe(DecisionTreeClassifier(class_weight="balanced", random_state=seed)),
            {"model__max_depth": [3, 5, 8],
             "model__min_samples_leaf": [5, 20, 50]}
        ),
        "random_forest": (
            pipe(RandomForestClassifier(class_weight="balanced", random_state=seed, n_jobs=1)),
            {"model__n_estimators": [300],     # 1 x 3 x 3 = 9 combinations
             "model__max_depth": [6, 10, None],
             "model__min_samples_leaf": [1, 5, 10]}
        ),
    }

In [43]:
candidates = build_candidates(recipe, cfg.random_state)
list(candidates)

['baseline', 'logreg', 'decision_tree', 'random_forest']

## Tune and compare with the same cross-validation

**Why one shared CV:** every model sees identical folds, so the comparison is fair. `StratifiedKFold` keeps the churn rate equal in each fold.

**Why several metrics:** accuracy is misleading on imbalanced data. We pick the winner by **ROC-AUC** (`primary_metric`) and also record average precision, recall, precision and F1.

**Why `refit=primary_metric`:** after the search, the best configuration is automatically re-fitted on the whole of `X_train`.

In [44]:
SCORING = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "recall": make_scorer(recall_score, zero_division=0),
    "precision": make_scorer(precision_score, zero_division=0),   # no warning if a model predicts no churners
    "f1": make_scorer(f1_score, zero_division=0),
}


def tune_and_compare(candidates: dict, X_train, y_train, cfg: Config):
    """Tune every candidate with the same CV on TRAIN data only; return a ranked table."""
    cv = StratifiedKFold(n_splits=cfg.cv_folds, shuffle=True, random_state=cfg.random_state)
    searches, rows = {}, []
    for name, (pipe, grid) in candidates.items():
        t0 = time.time()
        gs = GridSearchCV(pipe, grid, cv=cv, scoring=SCORING,
                          refit=cfg.primary_metric, n_jobs=-1)
        gs.fit(X_train, y_train)          # refit=True -> best pipeline refit on ALL of X_train
        i = gs.best_index_
        rows.append({
            "model": name,
            **{m: gs.cv_results_[f"mean_test_{m}"][i] for m in SCORING},
            f"{cfg.primary_metric}_std": gs.cv_results_[f"std_test_{cfg.primary_metric}"][i],
            "best_params": gs.best_params_,
            "seconds": round(time.time() - t0, 1),
        })
        searches[name] = gs
        log.info("%-14s CV %s=%.3f", name, cfg.primary_metric, rows[-1][cfg.primary_metric])
    table = pd.DataFrame(rows).sort_values(cfg.primary_metric, ascending=False).reset_index(drop=True)
    return searches, table

In [45]:
searches, comparison = tune_and_compare(candidates, X_train, y_train, cfg)
comparison.drop(columns="best_params").round(3)

2026-10-07 13:08:52,840 INFO baseline       CV roc_auc=0.500
2026-10-07 13:09:08,238 INFO logreg         CV roc_auc=0.846
2026-10-07 13:09:11,132 INFO decision_tree  CV roc_auc=0.830
2026-10-07 13:09:41,135 INFO random_forest  CV roc_auc=0.849


,model,roc_auc,average_precision,recall,precision,f1,roc_auc_std,seconds
0,random_forest,0.849,0.667,0.768,0.545,0.637,0.006,30.0
1,logreg,0.846,0.658,0.801,0.519,0.630,0.005,15.4
2,decision_tree,0.830,0.606,0.797,0.504,0.616,0.007,2.9
3,baseline,0.500,0.266,0.000,0.000,0.000,0.000,14.8


In [46]:
best_name = comparison.loc[0, "model"]
best_pipeline = searches[best_name].best_estimator_      # already refit on all of X_train
print("Selected:", best_name, "|", comparison.loc[0, "best_params"])

Selected: random_forest | {'model__max_depth': None, 'model__min_samples_leaf': 10, 'model__n_estimators': 300}


In [47]:
def choose_threshold(best_pipeline, X_train, y_train, cfg: Config) -> float:
    """Pick the decision threshold from out-of-fold TRAIN predictions (never from test).

    The default 0.5 is rarely right for an imbalanced problem. Out-of-fold means every
    prediction comes from a model that did not see that row, so the choice is honest.
    Swap f1_score for a business-cost function if misses and false alarms cost differently.
    """
    cv = StratifiedKFold(n_splits=cfg.cv_folds, shuffle=True, random_state=cfg.random_state)
    oof = cross_val_predict(clone(best_pipeline), X_train, y_train, cv=cv,
                            method="predict_proba", n_jobs=-1)[:, 1]
    grid = np.linspace(0.10, 0.90, 81)
    scores = [f1_score(y_train, oof >= t) for t in grid]
    return float(grid[int(np.argmax(scores))])

In [48]:
threshold = choose_threshold(best_pipeline, X_train, y_train, cfg)
print("Threshold:", round(threshold, 2))


Threshold: 0.49


In [49]:
def evaluate_on_test(pipeline, threshold: float, X_test, y_test) -> dict:
    """The ONE and only use of the test set."""
    proba = pipeline.predict_proba(X_test)[:, 1]
    pred = (proba >= threshold).astype(int)
    return {
        "threshold": threshold,
        "roc_auc": roc_auc_score(y_test, proba),
        "average_precision": average_precision_score(y_test, proba),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "confusion_matrix [[TN, FP], [FN, TP]]": confusion_matrix(y_test, pred).tolist(),
    }

In [50]:
test_metrics = evaluate_on_test(best_pipeline, threshold, X_test, y_test)
pd.Series(test_metrics)

threshold                                                   0.49
roc_auc                                                 0.836051
average_precision                                       0.635834
precision                                               0.520906
recall                                                  0.799465
f1                                                      0.630802
confusion_matrix [[TN, FP], [FN, TP]]    [[758, 275], [75, 299]]
dtype: object

## Step 10: Save the deployable artifact and its metadata
`
**Why:** `churn_pipeline.pkl` takes **raw** customer data and returns a probability, so the API cannot apply different preprocessing from training (no train/serve skew). The JSON records parameters, threshold, metrics, expected input columns and library versions so the run can be audited and reproduced.

In [51]:
def save_artifacts(best_name, pipeline, threshold, cv_row, test_metrics,
                   comparison, X_train, cfg: Config) -> None:
    """Persist the model together with the facts needed to trust and reproduce it."""
    cfg.model_dir.mkdir(parents=True, exist_ok=True)
    cfg.report_dir.mkdir(parents=True, exist_ok=True)

    joblib.dump(pipeline, cfg.model_dir / "churn_pipeline.pkl")
    comparison.to_csv(cfg.report_dir / "model_comparison.csv", index=False)

    metadata = {
        "model": best_name,
        "best_params": cv_row["best_params"],
        "decision_threshold": threshold,
        "cv_metrics": {k: cv_row[k] for k in SCORING},
        "test_metrics": test_metrics,
        "n_train_rows": len(X_train),
        "input_columns": list(X_train.columns),     # contract the API must satisfy
        "random_state": cfg.random_state,
        "trained_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "python": platform.python_version(),
        "sklearn": sklearn.__version__,
    }
    (cfg.report_dir / "model_metadata.json").write_text(json.dumps(metadata, indent=2, default=str))
    log.info("saved pipeline + metadata to %s and %s", cfg.model_dir, cfg.report_dir)

In [52]:
save_artifacts(best_name, best_pipeline, threshold, comparison.loc[0].to_dict(),
               test_metrics, comparison, X_train, cfg)

2026-10-07 13:09:47,888 INFO saved pipeline + metadata to D:\Projects\notebooks\Churn_Prediction_Project\models and D:\Projects\notebooks\Churn_Prediction_Project\reports


## Step 11: Smoke test the saved artifact on raw data

**Why:** confirms that the file we ship loads and accepts raw input exactly as the API will send it. This is a format check, not an evaluation.

In [53]:
loaded = joblib.load(cfg.model_dir / "churn_pipeline.pkl")
sample = X_test.head(3)                                   # raw rows: text categories, unscaled numbers
proba = loaded.predict_proba(sample)[:, 1]
print(pd.DataFrame({"churn_probability": proba.round(3), "flag": proba >= threshold}))

   churn_probability   flag
0              0.032  False
1              0.809   True
2              0.035  False
